# ImageNet Classification for the MLA100 NPU chip


## Load Dataset

In [1]:
!cp /content/drive/MyDrive/NPULab/spring-2026-data/imagenet_train20.txt imagenet_train20.txt
!cp /content/drive/MyDrive/NPULab/spring-2026-data/imagenet_val20.txt imagenet_val20.txt

# Text contains:
# file class_number
# file2 class_number2
# ...

In [2]:
!cp /content/drive/MyDrive/NPULab/spring-2026-data/imagenet_train20.zip imagenet_train20.zip

In [3]:
!cp /content/drive/MyDrive/NPULab/spring-2026-data/imagenet_val20.zip imagenet_val20.zip

In [4]:
!unzip -q imagenet_train20.zip
#   inflating: imagenet_train20a/n04346328/n04346328_2302.JPEG
#  inflating: imagenet_train20a/n04346328/n04346328_2842.JPEG

In [5]:
!unzip -q imagenet_val20.zip

## Install Requirements

In [6]:
!pip -q install onnxscript onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.3/159.3 kB 19.6 MB/s eta 0:00:00


In [7]:
# Import required libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torch.onnx
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os
import onnxscript
import onnxruntime as ort

## Prepare Dataset Loader

In [8]:
IMAGE_ROOT = 'imagenet_train20a'
TRAIN_LIST = 'imagenet_train20.txt'
BATCH_SIZE = 32
NUM_CLASSES = 20
INPUT_SHAPE = (240, 240)
NUM_EPOCHS = 1
LEARNING_RATE = 0.01

In [9]:
class ImageNet20Dataset(Dataset):
    def __init__(self, txt_file, root_dir, transform=None):
        self.img_labels = []
        self.root_dir = root_dir
        self.transform = transform

        with open(txt_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 2:
                    self.img_labels.append((parts[0], int(parts[1])))

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        filename, label = self.img_labels[idx]

        folder_name = filename.split('_')[0]

        full_path = os.path.join(self.root_dir, folder_name, filename)

        try:
            image = Image.open(full_path).convert("RGB")
        except Exception as e:
            print(f"Error loading {full_path}: {e}")
            image = Image.new('RGB', INPUT_SHAPE)

        if self.transform:
            image = self.transform(image)

        return image, label

## Prepare SimpleDNN

In [10]:
import torch
import torch.nn as nn

class SimpleDNN(nn.Module):
    def __init__(self, num_classes=20):
        super(SimpleDNN, self).__init__()

        self.features = nn.Sequential(
            # 240 → 120
            nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1, bias=True),
            nn.ReLU(inplace=False),

            # 120 → 60
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1, bias=True),
            nn.ReLU(inplace=False),

            # 60 → 30
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1, bias=True),
            nn.ReLU(inplace=False),
        )

        # Replace Linear(57600 → 20)
        # With Conv2d(64 → 20)
        self.classifier = nn.Conv2d(
            64,
            num_classes,
            kernel_size=1,
            stride=1,
            padding=0,
            bias=True
        )

        # Deterministic spatial reduction
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x):
        x = self.features(x)           # (1,64,30,30)
        x = self.classifier(x)         # (1,20,30,30)
        x = self.global_pool(x)        # (1,20,1,1)
        x = x.squeeze(3).squeeze(2)    # (1,20)
        return x


## Data Augmentations

In [11]:
transform = transforms.Compose([
    transforms.Resize(INPUT_SHAPE),
    transforms.ToTensor(),
])

try:
    train_dataset = ImageNet20Dataset(txt_file=TRAIN_LIST, root_dir=IMAGE_ROOT, transform=transform)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    print(f"Dataset loaded: {len(train_dataset)} images found.")
except Exception as e:
    print(f"Dataset setup skipped. Error: {e}")
    train_loader = []

Dataset loaded: 6000 images found.


## Train Model

### Intialize model, criterion and optimizer

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = SimpleDNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE)

Using device: cuda


### Start training loop

In [13]:
if len(train_loader) > 0:
    for epoch in range(NUM_EPOCHS):
        model.train() # Set model to training mode
        running_loss = 0.0

        for i, (images, labels) in enumerate(train_loader):
            # Move data to GPU if available
            images, labels = images.to(device), labels.to(device)

            # Zero the parameter gradients
            optimizer.zero_grad()

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Backward pass and optimize
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            # Print every 10 batches
            if (i + 1) % 10 == 0:
                print(f"Epoch [{epoch+1}/{NUM_EPOCHS}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}")

        epoch_loss = running_loss / len(train_loader)
        print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] Finished. Average Loss: {epoch_loss:.4f}")
else:
    print("Skipping training loop (Dataset empty or not found).")

Epoch [1/1], Step [10/188], Loss: 2.9970
Epoch [1/1], Step [20/188], Loss: 3.0163
Epoch [1/1], Step [30/188], Loss: 3.0026
Epoch [1/1], Step [40/188], Loss: 2.9938
Epoch [1/1], Step [50/188], Loss: 2.9873
Epoch [1/1], Step [60/188], Loss: 3.0279
Epoch [1/1], Step [70/188], Loss: 3.0242
Epoch [1/1], Step [80/188], Loss: 2.9986
Epoch [1/1], Step [90/188], Loss: 3.0004
Epoch [1/1], Step [100/188], Loss: 2.9996
Epoch [1/1], Step [110/188], Loss: 2.9999
Epoch [1/1], Step [120/188], Loss: 2.9825
Epoch [1/1], Step [130/188], Loss: 3.0048
Epoch [1/1], Step [140/188], Loss: 2.9856
Epoch [1/1], Step [150/188], Loss: 2.9810
Epoch [1/1], Step [160/188], Loss: 2.9827
Epoch [1/1], Step [170/188], Loss: 2.9913
Epoch [1/1], Step [180/188], Loss: 2.9819
Epoch [1/1] Finished. Average Loss: 2.9987


## Export Model to ONNX

In [17]:
model.eval()

# Move dummy input to same device as model
dummy_input = torch.randn(1, 3, 240, 240).to(device)
onnx_filename = "model.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_filename,
    verbose=False,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)

print(f"Model successfully exported to {onnx_filename}")

/tmp/ipython-input-2075491432.py:7: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0213 21:31:52.555000 790 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0213 21:31:52.556000 790 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0213 21:31:52.558000 790 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). 

Model successfully exported to model.onnx


In [18]:
# confirm onnx runtime classifies correctly
import numpy as np
from PIL import Image
import torchvision.transforms as transforms

VAL_IMAGE_ROOT = 'imagenet_val20'
VAL_LIST = 'imagenet_val20.txt'

NUM_IMAGES = 10   # set None for full validation

# -----------------------------
# Load validation list
# -----------------------------
samples = []
with open(VAL_LIST, "r") as f:
    for line in f:
        fname, label = line.strip().split()
        samples.append((fname, int(label)))

if NUM_IMAGES is not None:
    samples = samples[:NUM_IMAGES]
    print(f"num samples:{len(samples)}")

# -----------------------------
# Image preprocessing
# -----------------------------
def load_image(filename):

    print(f"filename: {filename}")

    folder = filename.split('_')[0]
    full_path = os.path.join(VAL_IMAGE_ROOT, filename)

    img = Image.open(full_path).convert("RGB")
    img = img.resize((240, 240), Image.BILINEAR)

    arr = np.array(img, dtype=np.float32) / 255.0  # ToTensor()
    arr = arr.transpose(2, 0, 1)                  # HWC → CHW
    arr = np.expand_dims(arr, axis=0)             # NCHW

    return arr.astype(np.float32)

test_images = []
labels = []

for fname, label in samples:
    test_images.append(load_image(fname))
    labels.append(label)

num samples:10
filename: n04346328_14853.JPEG
filename: n03938244_16157.JPEG
filename: n02006656_11578.JPEG
filename: n03721384_30474.JPEG
filename: n04346328_14659.JPEG
filename: n04252225_12404.JPEG
filename: n02117135_7660.JPEG
filename: n03935335_1074.JPEG
filename: n02415577_2813.JPEG
filename: n02808440_21132.JPEG


In [19]:
# onnx runtime test

ort_session = ort.InferenceSession(onnx_filename)
ort_inputs = {ort_session.get_inputs()[0].name: test_images[0]}
ort_outs = ort_session.run(None, ort_inputs)
print(ort_outs)

# [array([[-0.19135858, -0.23010466, -0.18045916, -0.05543918,  0.40322092,
#        -0.0212889 , -0.01825383, -0.3939799 , -0.11758958, -0.10948016,
#        -0.23098505,  0.15080309, -0.18484347,  0.3017688 , -0.2633909 ,
#         0.18919851,  0.2816175 ,  0.11519835,  0.05758833,  0.59133565]],
#      dtype=float32)]

# convert ort_outs to array
ort_outs_arr = np.array(ort_outs)
print(ort_outs_arr.shape)
# (1, 1, 20)

[array([[-0.09229438, -0.01382601, -0.04319517,  0.12931691,  0.00387009,
        -0.07925645,  0.12480734, -0.04674335, -0.05319367,  0.06565897,
        -0.07513767,  0.03019654,  0.02397025,  0.10250731,  0.05295286,
        -0.03050736, -0.05383794, -0.09081655, -0.00098991,  0.09250549]],
      dtype=float32)]
(1, 1, 20)
